In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from res_estimate_utils import *
import stim
import numpy as np

from mqt.qmap.na.zoned import ZonedNeutralAtomArchitecture
from mqt.qmap.na.zoned import RoutingAgnosticCompiler, RoutingAwareCompiler

In [3]:
# NOTE: r(theta, pi/2) = ry(theta)
def find_na_moves(circuit):
    from qiskit.circuit.library import HGate, CXGate, SGate, CYGate

    # H = Ry(pi/2) * Z
    h_equiv = QuantumCircuit(1)
    h_equiv.z(0)
    # h_equiv.ry(np.pi/2, 0)
    h_equiv.r(np.pi/2,np.pi/2, 0)
    sel.add_equivalence(HGate(), h_equiv)

    # CX = H_target * CZ * H_target
    cx_equiv = QuantumCircuit(2)
    cx_equiv.z(1)
    # cx_equiv.ry(np.pi/2, 1)
    cx_equiv.r(np.pi/2, np.pi/2, 1)
    cx_equiv.cz(0, 1)
    cx_equiv.z(1)
    # cx_equiv.ry(np.pi/2, 1)
    cx_equiv.r(np.pi/2, np.pi/2, 1)
    sel.add_equivalence(CXGate(), cx_equiv)

    # S = Rz(pi/2), but Rz isn't in basis. S = Z^(1/2)
    # S can be decomposed as: Ry(0) then phase via Z gates
    # Actually: S = diag(1, i) = Rz(pi/2) up to global phase
    # Use: Rz(theta) = Z * Ry(-theta) * Z * Ry(theta) ... but simpler:
    # Just add 'rz' to basis gates since ry+z+rz is more natural
    s_equiv = QuantumCircuit(1)
    s_equiv.rz(np.pi/2, 0)
    sel.add_equivalence(SGate(), s_equiv)

    # CY = (S_dag on target) * CX * (S on target)
    cy_equiv = QuantumCircuit(2)
    cy_equiv.rz(-np.pi/2, 1)  # S_dag
    cy_equiv.z(1)
    cy_equiv.ry(np.pi/2, 1)
    cy_equiv.cz(0, 1)
    cy_equiv.z(1)
    cy_equiv.ry(np.pi/2, 1)
    cy_equiv.rz(np.pi/2, 1)   # S
    sel.add_equivalence(CYGate(), cy_equiv)

    transpiled_qc = transpile(circuit, basis_gates=['r', 'rz', 'z', 'cz'], optimization_level=1)


    arch = ZonedNeutralAtomArchitecture.from_json_string("""{
    "name": "Architecture with one entanglement and one storage zone",
    "operation_duration": {"rydberg_gate": 0.36, "single_qubit_gate": 52, "atom_transfer": 15},
    "operation_fidelity": {"rydberg_gate": 0.995, "single_qubit_gate": 0.9997, "atom_transfer": 0.999},
    "qubit_spec": {"T": 1.5e6},
    "storage_zones": [{
        "zone_id": 0,
        "slms": [{"id": 0, "site_separation": [3, 3], "r": 20, "c": 100, "location": [0, 0]}],
        "offset": [0, 0],
        "dimension": [297, 57]
    }],
    "entanglement_zones": [{
        "zone_id": 0,
        "slms": [
        {"id": 1, "site_separation": [12, 10], "r": 7, "c": 20, "location": [35, 67]},
        {"id": 2, "site_separation": [12, 10], "r": 7, "c": 20, "location": [37, 67]}
        ],
        "offset": [35, 67],
        "dimension": [230, 60]
    }],
    "aods": [{"id": 0, "site_separation": 2, "r": 100, "c": 100}],
    "rydberg_range": [[[30, 62], [270, 132]]]
    }""")

    compiler = RoutingAwareCompiler(arch)

    compiler_moves = calculate_movements_for_arch(arch, transpiled_qc, compiler)

    return compiler_moves


# Folded transversal Circuit

In [4]:
with open("folded_transversal_without_escape.txt", 'r') as f:
    circuit_str = f.read()

stim_circuit = stim.Circuit(circuit_str)
qiskit_circuit, ft_moves = stim_to_qiskit(stim_circuit, return_meas_reset_moves=True)

In [5]:
print(f"Number of qubits: {qiskit_circuit.num_qubits} depth: {qiskit_circuit.depth()}")

Number of qubits: 131 depth: 34


In [6]:
folded_transversal_compiler_moves = find_na_moves(qiskit_circuit)

ValueError: [1;31m[ERROR][0m Unsupported single-qubit gate: r


In [7]:
print(f"Compiler predicted move: {folded_transversal_compiler_moves}, (reset+meas)moves: {ft_moves}")
print(f"Total: {folded_transversal_compiler_moves+ft_moves}(moves)")

Compiler predicted move: 510, (reset+meas)moves: 12
Total: 522(moves)


# HyperCube

In [4]:
with open("h6_with_escape.txt", 'r') as f:
    circuit_str = f.read()

stim_circuit = stim.Circuit(circuit_str).without_noise()
qiskit_circuit, h6_moves = stim_to_qiskit(stim_circuit, return_meas_reset_moves=True)

In [5]:
print(f"Number of qubits: {qiskit_circuit.num_qubits} depth: {qiskit_circuit.depth()}")

Number of qubits: 1272 depth: 55


In [10]:
h6_compiler_moves = find_na_moves(qiskit_circuit)

In [11]:
print(f"Compiler predicted move: {h6_compiler_moves}, (reset+meas)moves: {h6_moves}")
print(f"Total: {h6_compiler_moves+h6_moves}(moves)")

Compiler predicted move: 1148, (reset+meas)moves: 0
Total: 1148(moves)
